In [ ]:
!pip install crewai crewai_tools

In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

# Importing modules, LLMs and libraries

In [ ]:
import os
from google.colab import userdata
os.environ["OPENAI_MODEL_NAME"] = "gpt-4o-mini"
os.environ["OPENAI_API_KEY"]=userdata.get('OPENAI_API_KEY')
os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY')

In [ ]:
from crewai import Agent, Task, Crew

# Creating Agents
Creating 3 agents

*   Venue Coordinator
*   Logistics Manager
*   Marketing and Communications Agent



## crewAI Tools

In [ ]:
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

# Initialize the tools
search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

## Agent 1: Venue Coordinator

In [ ]:
venue_coordinator = Agent(
    role="Venue Coordinator",
    goal="Identify and book an appropriate venue "
    "based on event requirements",
    backstory=(
        "With a keen sense of space and "
        "understanding of event logistics, "
        "you excel at finding and securing "
        "the perfect venue that fits the event's theme, "
        "size, and budget constraints."
    ),
    tools=[search_tool, scrape_tool],
    verbose=True
)

## Agent 2: Logistics Manager

In [ ]:
logisitics_manager = Agent(
    role="Logistics Manager",
    goal=(
        "Manage all logistics for the event "
        "including catering and equipment"
    ),
    backstory=(
        "Organized and detail-oriented, "
        "you ensure that every logistical aspect of the event "
        "from catering to equipment setup "
        "is flawlessly executed to create a seamless experience."
    ),
    tools=[search_tool, scrape_tool],
    verbose=True
)

# Agent 3: Marketing and Communication Agent

In [ ]:
marketing_communications_agent = Agent(
    role="Marketing and Communications Agent",
    goal="Effectively market the event and "
         "communicate with participants",
    backstory=(
        "Creative and communicative, "
        "you craft compelling messages and "
        "engage with potential attendees "
        "to maximize event exposure and participation."
    ),
    tools=[search_tool, scrape_tool],
    verbose=True
)

# Step 03. Creating Tasks
Here, we will be creating 3 tasks


*   Venue Task (done by Venue Coordinator)
*   Logistics Task (done by Logistics Manager) - Async Task
*   Marketing Task (done by Marketing and Communications Agent) - Async Task



# Creating VenueDetails Pydantic Model


*   Create a class VenueDetails using Pydantic BaseModel.
*   Agents will populate this object with information about different venues    by creating different instances of it



In [ ]:
from pydantic import BaseModel

In [ ]:
class VenueDetails(BaseModel):
    name: str
    address: str
    capacity: int
    booking_status: str

# Creating Tasks


*   By using output_json, you can specify the structure of the output you want.
*   By using output_file, you can get your output in a file.
*   By setting **human_input=True**, the task will ask for human feedback (whether you like the results or not) before finalising it.

## Task 1: Venue Task



In [ ]:
venue_task = Task(
    description="Find a venue in {event_city} "
                "that meets criteria for {event_topic}.",
    expected_output="All the details of a specifically chosen"
                    "venue you found to accommodate the event.",
    output_json=VenueDetails,
    output_file="venue_details.json",  # Outputs the venue details as a JSON file
    agent=venue_coordinator,
    human_input=True
)

## Task 2: Logistics Task


*   By setting **async_execution=True**, it means the task can run in parallel with the tasks which come after it.




In [ ]:
logsitics_task = Task(
    description="Coordinate catering and "
                 "equipment for an event "
                 "with {expected_participants} participants "
                 "on {tentative_date}.",
    expected_output="Confirmation of all logistics arrangements "
                    "including catering and equipment setup.",
    agent=logisitics_manager,
    human_input=True,
    #async_execution=True,
)

## Task 3: Marketing Task


*   By setting **async_execution=True**, it means the task can run in parallel with the tasks which come after it.




In [ ]:
marketing_task = Task(
    description="Promote the {event_topic} "
                "aiming to engage at least"
                "{expected_participants} potential attendees.",
    expected_output="Report on marketing activities "
                    "and attendee engagement formatted as markdown.",
    agent=marketing_communications_agent,
    async_execution=True,
    output_file="marketing_report.md" # Outputs the report as a text file
)

## Creating Crew
**Note:** Since you set async_execution=True for logistics_task and marketing_task tasks, now the order for them does not matter in the tasks list.


In [ ]:
crew = Crew(
    agents=[venue_coordinator, logisitics_manager, marketing_communications_agent],
    tasks=[venue_task, logsitics_task, marketing_task],
    verbose=True
)

## Running the Crew
Setting the inputs for execution of the crew

In [ ]:
event_details = {
    'event_topic': "Tech Innovation Conference",
    'event_description': "A gathering of tech innovators "
                         "and industry leaders "
                         "to explore future technologies.",
    'event_city': "San Francisco",
    'tentative_date': "2024-09-15",
    'expected_participants': 500,
    'budget': 20000,
    'venue_type': "Conference Hall"
}


## **Note 2:**

Since you set **human_input=True** for some tasks, the execution will ask for your input before it finishes running.

When it asks for feedback, use your mouse pointer to first click in the text box before typing anything.

In [ ]:
results = crew.kickoff(inputs=event_details)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  161626da-04b2-438e-85e4-6f0ae024a2e7                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Find a venue in San Francisco that meets criteria for Tech Innovation Conference.                        │
│  ID: ffc16152-bfab-4634-a9cd-158850da0184                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Venue Coordinator                                                                                       │
│                                                                                                                 │
│  Task: Find a venue in San Francisco that meets criteria for Tech Innovation Conference.                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#4) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: Search the internet with Serper                                                                          │
│  Args: {"search_query": "Tech Innovation Conference venue San Francisco"}                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  {'searchParameters': {'q': 'Tech Innovation Conference venue San Francisco', 'type': 'search', 'num': 10,      │
│  'engine': 'google'}, 'organic': [{'title': 'AI Industry Conference Venue in SF', 'link':                       │
│  'https://ssfconf.com/ai-conference-venue-sf/', 'snippet': 'Located just minutes from San Francisco             │
│  International Airport, the South San Francisco Conference Center is the ideal venue for AI industry            │
│  conferences, expos, ...', 'position': 1}, {'title': 'World Agri-Tech Innovation Summit San Francisco - March   │
│  17-18 ...', 'link': 'https://worldagritechusa.com/', 'snippet': 'Meet the global leaders driving sustainable   │
│  agricultural practices in San Francisco on March 17–18, 2026. Join 2,000+ global leaders from agribusinesses,  │
│  ...', 'position': 2}, {'title': '2026 USA - RSAC Conference', 'link': 'https://www.rsaconference.com/usa',     │
│  'snippet': 'San Francisco. Moscone Center | March 23 - 26, 2026. The Power of Community ... Innovation         │
│  Programs; Expo & Sponsor Briefings. $2,395. Dec. 6 - Feb. 20 ...', 'position': 3}, {'title': 'TECHSPO Silicon  │
│  Valley Technology ExpoTECHSPO Silicon Valley ...', 'link': 'https://techsposv.com/', 'snippet': 'Join us on    │
│  June 16 – 17, 2026, at the Grand Hyatt San Francisco Hotel in Silicon Valley for 2 unforgettable days of       │
│  technology exploration. TECHSPO brings ...', 'position': 4}, {'title': "Moscone Center: San Francisco's        │
│  Premier Convention Center", 'link': 'https://www.moscone.com/', 'snippet': 'Welcome to The Moscone Center!     │
│  The Moscone Center Administrative Office 747 Howard Street San Francisco, CA 94103 415.974.4000', 'position':  │
│  5}, {'title': 'San Francisco AI Event Space at The Midway', 'link':                                            │
│  'https://corporate.themidwaysf.com/san-francisco-event-space-ai/', 'snippet': "The Midway is San Francisco's   │
│  premier venue for AI events-Immersive spaces, turnkey AV, and proven expertise in tech and investor            │
│  gatherings.", 'position': 6}, {'title': 'SPIE Photonics West', 'link':                                         │
│  'https://spie.org/conferences-and-exhibitions/photonics-west', 'snippet':...                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#6) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: Read website content                                                                                     │
│  Args: {"website_url": "https://ssfconf.com/ai-conference-venue-sf/"}                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (26 outputs left out)

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  The following text is scraped website content:                                                                 │
│  Tech Innovation 2023 | SGInnovate                                                                              │
│  ×                                                                                                              │
│  About SGInnovate                                                                                               │
│  Our Mission                                                                                                    │
│  Growing Singapore’s Deep Tech Landscape                                                                        │
│  Our Leadership                                                                                                 │
│  Board Members & Management                                                                                     │
│  Our Partners                                                                                                   │
│  SGInnovate’s Global Network                                                                                    │
│  Press Room                                                                                                     │
│  Latest Announcements                                                                                           │
│  Need Help?                                                                                                     │
│  Contact Us                                                                                                     │
│  FAQs                                                                                                           │
│  What We Do                                                                                                     │
│  Deep Tech Central                                                                                              │
│  Our ecosystem approach to developing talent                                                                    │
│  Deep Tech Community                                                                                            │
│  Learn more about emerging tech trends and opportunities                                                        │
│  Career Development & Jobs                                                                                      │
│  Discover Deep Tech jobs and development programmes                                                             │
│  Startup Growth & Funding                                                                                       │
│  Access resources and support dedicated to Deep Tech companies                                                  │
│  Need Help?                                                                                                     │
│  Contact Us                                                                                                     │
│  FAQs                                                                                                           │
│  Our Investments                                                                                                │
│  Our Portfolio                                                                                                  │
│  Explore our portfolio companies                                                                                │
│  Our Approach                                                                                                   │
│  Building startups at the frontiers of emerging tech                                                            │
│  Our Team                                             

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Marketing and Communications Agent                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Tech Innovation Conference 2023 Marketing Report                                                             │
│                                                                                                                 │
│  **Event Overview**                                                                                             │
│                                                                                                                 │
│  - **Conference Name:** Tech Innovation 2023                                                                    │
│  - **Dates:** October 31 - November 2, 2023                                                                     │
│  - **Location:** Marina Bay Sands, 10 Bayfront Ave, Singapore                                                   │
│  - **Theme:** Sustainable Urban Living, with a focus on AI in Healthcare.                                       │
│                                                                                                                 │
│  **Daily Themes:**                                                                                              │
│  - **Day 1:** Sustainability                                                                                    │
│  - **Day 2:** Health & Well-Being                                                                               │
│  - **Day 3:** AI in Healthcare (Main focus)                                                                     │
│                                                                                                                 │
│  **Engagement Objectives:**                                                                                     │
│  - Target at least 500 potential attendees from various sectors including Health and Biomedical Sciences,       │
│  MedTech, and Digital Health.                                                                                   │
│                                                                                                                 │
│  **Outreach Strategies:**                                                                                       │
│  1. **Email Campaigns:**                                                                                        │
│     - Target relevant industry contacts providing them with information about the event.                        │
│     - Highlight keynote speakers and significant opportunities for networking.                                  │
│                                                                                                                 │
│  2. **Social Media Promotions:**                                                                                │
│     - Utilize platforms like LinkedIn and Twitter to share updates, speaker highlights, and conference          │
│  insights.                                                                                                      │
│     - Run ads targeting professionals interested in technology and healthcare.                                  │
│                                                                                                                 │
│  3. **Partnerships:**                                                                                           │
│     - Collaborate with industry organizations to promot

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Promote the Tech Innovation Conference aiming to engage at least500 potential attendees.                       │
│  Agent:                                                                                                         │
│  Marketing and Communications Agent                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  161626da-04b2-438e-85e4-6f0ae024a2e7                                                                           │
│  Final Output: # Tech Innovation Conference 2023 Marketing Report                                               │
│                                                                                                                 │
│  **Event Overview**                                                                                             │
│                                                                                                                 │
│  - **Conference Name:** Tech Innovation 2023                                                                    │
│  - **Dates:** October 31 - November 2, 2023                                                                     │
│  - **Location:** Marina Bay Sands, 10 Bayfront Ave, Singapore                                                   │
│  - **Theme:** Sustainable Urban Living, with a focus on AI in Healthcare.                                       │
│                                                                                                                 │
│  **Daily Themes:**                                                                                              │
│  - **Day 1:** Sustainability                                                                                    │
│  - **Day 2:** Health & Well-Being                                                                               │
│  - **Day 3:** AI in Healthcare (Main focus)                                                                     │
│                                                                                                                 │
│  **Engagement Objectives:**                                                                                     │
│  - Target at least 500 potential attendees from various sectors including Health and Biomedical Sciences,       │
│  MedTech, and Digital Health.                                                                                   │
│                                                                                                                 │
│  **Outreach Strategies:**                                                                                       │
│  1. **Email Campaigns:**                                                                                        │
│     - Target relevant industry contacts providing them with information about the event.                        │
│     - Highlight keynote speakers and significant opportunities for networking.                                  │
│                                                                                                                 │
│  2. **Social Media Promotions:**                                                                                │
│     - Utilize platforms like LinkedIn and Twitter to share updates, speaker highlights, and conference          │
│  insights.                                                                                                      │
│     - Run ads targeting professionals interested in technology and healthcare.                                  │
│                                                       